# Trabajo Fin de Máster  
### Análisis de la Ciudad mediante Aprendizaje Supervisado  
#### Detección Automática de Tipologías Residenciales y Patrones de Cerramiento: Interpretabilidad vs Rendimiento

**Master Universitario en Ciencia de Datos e Ingeniería de Computadores (Universidad de Granada)**

> **Autor:** David Fernández Martínez    
> **Email personal:** david.fernxndez.martinez@gmail.com  
> **Email académico:** davidfm8@correo.ugr.es  
> **LinkedIn:** [linkedin.com/in/david-fernández-martínez](https://www.linkedin.com/in/david-fern%C3%A1ndez-mart%C3%ADnez/)  
> **GitHub:** [github.com/davidfernxndez](https://github.com/davidfernxndez)

---

## Metodología para el análisis de interpretabilidad global

### 📝 Descripción del notebook
En este notebook se presenta la metodología y mecanismos empleados para realizar el análisis de interpretabilidad global de modelos transparantes (Regresión logística y árbol de decisión) y modelos caja negra (SVM, Random Forest y XGBoost).

El análisis de interpretabilidad global, de acuerdo a la metodología descrita en este *notebook*, se realiza en el *notebook* [4.2_Global_interp_analysis.ipynb](4.2_Global_interp_analysis.ipynb).

### Indice de contenidos
1. [Enfoque de interpretabilidad global](#interp_global)

2. [Entrenamiento de modelos para interpretabilidad](#train)

3. [Métodos y técnicas de interpretabilidad](#metodos)
    * [3.1. Mecanismos intrínsecos en modelos transparantes](#intrinsecos)
    * [3.2 *SHAP (SHapley Additive exPlanations)*](#shap)

# Configuración de entorno e *imports*

Este proyecto ha sido realizado en un entorno Anaconda con la versión 3.11.15 de *Python*. Las versiones de las librerias requeridas se encuentran en el fichero *requirements-full.txt*.

En esta sección se importan las librerias necesarias para la ejecución de este fichero *jupyter notebook*, se activa el *reload* de módulos externos y se configuran aspectos globales y de reproducibilidad.

In [2]:
# jupyter extensions to automatically reload external modules
%load_ext autoreload
%autoreload 2

In [1]:
import warnings
import random
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from src.balanced_xgb import BalancedXGBClassifier

# Configuration object
from src.config import cfg

# Production training method
from src.production_training import train_final_model

**Solución a problemas en *imports***

Si los *imports* del módulo `src` fallan al ejecutar este cuaderno en un entorno diferente, descomente y ejecute la siguiente celda:

```python
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [3]:
# Global configuration
sns.set_theme(style="ticks", context="notebook")
plt.rcParams["font.family"] = "sans-serif"
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
warnings.filterwarnings("ignore")

In [4]:
# Reproducibility
SEED = cfg.SEED 
np.random.seed(SEED)
random.seed(SEED)

<a id="interp_global"></a>
# 1. Enfoque de interpretabilidad global

El análisis de interpretabilidad global tiene como objetivo comprender el comportamiento general de cada modelo mediante la identificación de las variables con mayor influencia sobre sus predicciones. Dada la naturaleza multiclase del problema, el análisis se desagrega además para cada grado de cerramiento. De este modo, es posible identificar los factores que el modelo considera más relevantes para distinguir cada categoría y obtener una caracterización urbanística de los distintos grados de cerramiento.

Este análisis se aplica a todos los modelos considerados en el experimento con el fin de obtener una visión global y comparativa de su comportamiento. Por un lado se pretende determinar si los diferentes algoritmos basan sus predicciones en un conjunto similar de variables relevantes o si, por el contrario, identifican patrones diferenciados en los datos. Por otro lado, este análisis permite evaluar hasta qué punto el mecanismo de explicabilidad empleado facilita la transferencia de conocimiento hacia los expertos del dominio mediante explicaciones adecuadas y comprensibles.

En el protocolo experimental de evaluación de rendimiento, presentado en el *notebook* [3.1_Performance_methodology.ipynb](3.1_Performance_methodology.ipynb), se ha empleado la estrategia *Nested Cross-Validation* con el objetivo de obtener una estimación robusta de la capacidad de generalización de los distintos algoritmos. Como resultado de este procedimiento se obtienen varios modelos, cada uno entrenado sobre un conjunto de entrenamiento *Outer Train Set* diferente y con una configuración de hiperparámetros específica.

Sin embargo, el estudio de la interpretabilidad requiere analizar un único modelo representativo. Para ello, una vez finalizada la evaluación de rendimiento, se obtiene la configuración óptima de hiperparámetros mediante una validación cruzada estándar sobre el conjunto de datos completo, explorando el mismo espacio de búsqueda definido para la evaluación del rendimiento. Con el fin de mantener la consistencia metodológica y garantizar la reproducibilidad del procedimiento, esta validación reutiliza las mismas particiones empleadas en el bucle externo de la *Nested Cross-Validation*.

Finalmente, cada modelo se entrena sobre todo el conjunto de datos disponible utilizando la configuración óptima de hiperparámetros. Este procedimiento reproduce el escenario habitual previo al despliegue en producción, donde, una vez definido el algoritmo y sus parámetros de configuración, se utiliza toda la información disponible para obtener la versión final del modelo. Por ello, el estudio de interpretabilidad se realiza sobre este modelo final, al representar la configuración que sería empleada para realizar predicciones en un entorno real.

<a id="train"></a>
# 2. Entrenamiento de modelos para interpretabilidad

Para llevar a cabo el proceso de optimización de hiperparámetros y entrenamiento sobre el conjunto de datos completo, se ha desarrollado la función *train_final_model*, ubicada en el módulo *src/production_training.py*.

A continuación se entrenan todos los modelos mediante esta función con la misma configuración de hiperparámetros que la utilizada en [3.1_Performance_methodology.ipynb](3.1_Performance_methodology.ipynb). Los modelos resultantes se almacenan en el directorio *output/models* en formato *.pkl*.

In [5]:
################################
# Multinomial Logistic Regression
################################

LR_model = LogisticRegression(
    class_weight="balanced",
    solver="lbfgs",
    penalty="l2",
    random_state = SEED
)


LR_param_grid = {
    "C": [10, 100, 1000, 10000],
}

LR_model = train_final_model(cfg, LR_model, LR_param_grid, "Logistic_Regression")


TRAIN MODEL FOR PRODUCTION ON ALL AVAILABLE DATA
Model Name      : Logistic_Regression
CV Folds        : 5

Hyperparameter Grid:
C                        : [10, 100, 1000, 10000]
Fitting 5 folds for each of 4 candidates, totalling 20 fits

--------------------------------------------------------------------------------
Total time      : 7.48 seconds
--------------------------------------------------------------------------------
Best params for Logistic_Regression:
{'C': 10}

--------------------------------------------------------------------------------
Model and Encoder saved in directory: C:\Users\david\Documents\MASTER_CIENCIA_DE_DATOS\TFM\PROJECT\output\models
--------------------------------------------------------------------------------


In [6]:
################################
# Decision Tree
################################

DT_model = DecisionTreeClassifier(
    class_weight = "balanced",
    max_depth = 6,
    random_state = SEED
)

DT_param_grid = {
    "max_leaf_nodes": [10, 15, 20, 25, 30],
    "min_samples_leaf": [5, 10, 15],
    "criterion": ["gini", "entropy"],
}
DT_model = train_final_model(cfg, DT_model, DT_param_grid, "Decision_Tree")


TRAIN MODEL FOR PRODUCTION ON ALL AVAILABLE DATA
Model Name      : Decision_Tree
CV Folds        : 5

Hyperparameter Grid:
max_leaf_nodes           : [10, 15, 20, 25, 30]
min_samples_leaf         : [5, 10, 15]
criterion                : ['gini', 'entropy']
Fitting 5 folds for each of 30 candidates, totalling 150 fits

--------------------------------------------------------------------------------
Total time      : 0.70 seconds
--------------------------------------------------------------------------------
Best params for Decision_Tree:
{'criterion': 'entropy', 'max_leaf_nodes': 15, 'min_samples_leaf': 5}

--------------------------------------------------------------------------------
Model and Encoder saved in directory: C:\Users\david\Documents\MASTER_CIENCIA_DE_DATOS\TFM\PROJECT\output\models
--------------------------------------------------------------------------------


In [7]:
################################
# SVM With RBF Kernel
################################

SVM_model = SVC(
    kernel = "rbf",
    decision_function_shape = 'ovr',
    class_weight = "balanced",
    random_state = SEED,
    probability=True
)

SVM_param_grid = {
    "C": [0.1, 1, 10, 100],
    "gamma": ["scale", "auto", 0.01, 0.1]
}

SVM_model = train_final_model(cfg, SVM_model, SVM_param_grid, "SVM")


TRAIN MODEL FOR PRODUCTION ON ALL AVAILABLE DATA
Model Name      : SVM
CV Folds        : 5

Hyperparameter Grid:
C                        : [0.1, 1, 10, 100]
gamma                    : ['scale', 'auto', 0.01, 0.1]
Fitting 5 folds for each of 16 candidates, totalling 80 fits

--------------------------------------------------------------------------------
Total time      : 1.69 seconds
--------------------------------------------------------------------------------
Best params for SVM:
{'C': 10, 'gamma': 'auto'}

--------------------------------------------------------------------------------
Model and Encoder saved in directory: C:\Users\david\Documents\MASTER_CIENCIA_DE_DATOS\TFM\PROJECT\output\models
--------------------------------------------------------------------------------


In [8]:
################################
# Random Forest
################################

RF_model = RandomForestClassifier(
        class_weight = "balanced_subsample",
        random_state = SEED
)


RF_param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [5, 10, None],
    "max_features": ["sqrt", 0.3, 0.4],
    "criterion": ['gini', 'entropy'],
    "min_samples_split": [2, 5],
}  

RF_model = train_final_model(cfg, RF_model, RF_param_grid, "Random_Forest")


TRAIN MODEL FOR PRODUCTION ON ALL AVAILABLE DATA
Model Name      : Random_Forest
CV Folds        : 5

Hyperparameter Grid:
n_estimators             : [100, 200, 300]
max_depth                : [5, 10, None]
max_features             : ['sqrt', 0.3, 0.4]
criterion                : ['gini', 'entropy']
min_samples_split        : [2, 5]
Fitting 5 folds for each of 108 candidates, totalling 540 fits

--------------------------------------------------------------------------------
Total time      : 65.69 seconds
--------------------------------------------------------------------------------
Best params for Random_Forest:
{'criterion': 'gini', 'max_depth': 10, 'max_features': 0.3, 'min_samples_split': 5, 'n_estimators': 100}

--------------------------------------------------------------------------------
Model and Encoder saved in directory: C:\Users\david\Documents\MASTER_CIENCIA_DE_DATOS\TFM\PROJECT\output\models
----------------------------------------------------------------------------

In [9]:
################################
# XGBoost
################################

XG_model = BalancedXGBClassifier(
        random_state = SEED,
        sampling_method = "uniform",
        objective= "multi:softmax",
        eval_metric="mlogloss",
)

XG_param_grid = {
    "n_estimators": [100, 300],
    "max_depth": [5, 10],
    "learning_rate": [0.01, 0.1],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0],
    "gamma": [0, 0.3],
    "reg_lambda": [1, 5]
}

XG_model = train_final_model(cfg, XG_model, XG_param_grid, "XGBoost")


TRAIN MODEL FOR PRODUCTION ON ALL AVAILABLE DATA
Model Name      : XGBoost
CV Folds        : 5

Hyperparameter Grid:
n_estimators             : [100, 300]
max_depth                : [5, 10]
learning_rate            : [0.01, 0.1]
subsample                : [0.8, 1.0]
colsample_bytree         : [0.8, 1.0]
gamma                    : [0, 0.3]
reg_lambda               : [1, 5]
Fitting 5 folds for each of 128 candidates, totalling 640 fits

--------------------------------------------------------------------------------
Total time      : 25.85 seconds
--------------------------------------------------------------------------------
Best params for XGBoost:
{'colsample_bytree': 0.8, 'gamma': 0, 'learning_rate': 0.01, 'max_depth': 5, 'n_estimators': 300, 'reg_lambda': 1, 'subsample': 0.8}

--------------------------------------------------------------------------------
Model and Encoder saved in directory: C:\Users\david\Documents\MASTER_CIENCIA_DE_DATOS\TFM\PROJECT\output\models
-------------

<a id="metodos"></a>
# 3. Métodos y técnicas de interpretabilidad

En esta sección se describen las herramientas empleadas para el análisis de interpretabilidad, detallando los mecanismos utilizados en función de la naturaleza de cada modelo y su aplicación dentro del estudio experimental.

<a id="intrinsecos"></a>
## 3.1 Mecanismos intrínsecos en modelos transparantes

Los modelos transparentes incorporan mecanismos de interpretabilidad propios de su estructura interna, por lo que el
estudio se basa en utilizar estos mecanismos que constituyen la razón principal para su inclusión en el diseño experimental.

**Coeficientes de regresión**

El modelo de regresión logística multinomial ofrece un elevado grado de interpretabilidad gracias a la relación directa existente entre las variables predictoras y los coeficientes $\beta$ aprendidos por el modelo. Estos coeficientes describen cómo cada variable influye sobre la probabilidad de pertenencia a las distintas categorías de la variable objetivo. Su interpretación se basa en el signo y magnitud de dichos coeficientes.

El signo del coeficiente indica la dirección del efecto que produce una variable predictora sobre la probabilidad de pertenencia a una determinada clase:

* $\beta > 0$. Un coeficiente positivo indica que la presencia de la característica incrementa la probabilidad de pertenencia a esa clase frente al resto de categorías. Estas variables pueden interpretarse como factores favorecedores o catalizadores de dicha clase.

* $\beta < 0$. Un coeficiente negativo indica que la presencia de la característica reduce la probabilidad de pertenencia a esa clase. Estas variables pueden interpretarse como factores inhibidores.

La magnitud del coeficiente refleja la intensidad de la contribución de una variable a la "probabilidad cruda" (*log-odd*), de pertenecer a cada clase. Dado que la regresión logística multinomial modela relaciones lineales y aditivas, cada coeficiente contribuye sumando o restando, al cálculo de la probabilidad estimada. Por lo tanto, coeficientes con magnitudes elevadas, tanto positivas como negativas, indican una mayor influencia de la variable sobre la predicción del modelo, mientras que valores próximos a cero reflejan una capacidad discriminativa baja.

La interpretación de la magnitud de los coeficientes depende de la escala de las variables predictoras. En este conjunto de datos, todas las variables son categórico binarias, lo que permite comparar directamente los coeficientes entre sí. La interpretación se centra en analizar el efecto que produce la presencia o ausencia de una determinada característica, manteniendo constantes el resto de variables.

El análisis de interpretabilidad basado en los coeficientes permite estudiar el comportamiento del modelo desde dos perspectivas complementarias:

* **Importancia global de variables**. Permite identificar aquellas características con mayor influencia sobre las predicciones, calculando la magnitud media de sus coeficientes en valor absoluto considerando todas las clases.

* **Análisis por clase**. Permite caracterizar cada grado de cerramiento mediante las variables que más contribuyen a su predicción, teniendo en cuenta además el signo del coeficiente para determinar si su efecto favorece o reduce la probabilidad de pertenencia a dicha categoría.

**Reglas de decisión y reducción de impureza**

La interpretabilidad del árbol de decisión se basa en dos mecanismos intrínsecos derivados de su propia estructura: la importancia de las variables mediante la reducción de impureza y el análisis de las reglas de decisión.

La importancia de una variable se determina a partir de la reducción acumulada de impureza producida por dicha variable en los nodos donde interviene para dividir el conjunto de datos. Por tanto, aquellas variables que generan particiones más homogéneas respecto a la clase objetivo presentan una mayor contribución al proceso de decisión del árbol. Este mecanismo proporciona una visión global del comportamiento del modelo, permitiendo identificar las características en las que se fundamentan sus predicciones.

Para profundizar en la explicabilidad del modelo, se estudia la estructura interna del árbol mediante la extracción de las reglas de decisión asociadas a sus hojas terminales. Cada recorrido desde el nodo raíz hasta una hoja define una regla de clasificación que puede expresarse como: "si se cumplen las condiciones $x_1, x_2, \dots, x_n$, entonces la observación se asigna a la clase $k$". Estas reglas permiten identificar las combinaciones de características utilizadas por el árbol para diferenciar las distintas categorías. Además, la distribución de clases en cada hoja permite evaluar el grado de determinismo de las reglas obtenidas. Las hojas con una elevada concentración de observaciones pertenecientes a una única clase representan reglas con una mayor capacidad discriminativa, mientras que aquellas con distribuciones más equilibradas reflejan situaciones de mayor ambigüedad en la clasificación.

En consecuencia, la calidad de la interpretabilidad proporcionada por el árbol de decisión depende de que las reglas extraídas representen relaciones coherentes con el conocimiento del dominio y permitan distinguir adecuadamente entre las distintas categorías.

<a id="shap"></a>
## 3.2 *SHAP (SHapley Additive exPlanations)*

Para analizar el comportamiento de los modelos caja negra (*SVM*, *Random Forest* y *XGBoost*) se emplea la técnica de explicabilidad *post-hoc* *SHAP (SHapley Additive exPlanations)*. Esta técnica permite aproximar el proceso de decisión de modelos complejos mediante la cuantificación de la contribución individual de cada variable a las predicciones realizadas.

*SHAP* pertenece a la familia de métodos de atribución aditiva de características, ya que representa una predicción como la suma de las contribuciones individuales de cada variable de entrada. De este modo, permite cuantificar el efecto de cada característica sobre la desviación de la predicción respecto a un valor de referencia denominado valor base, que corresponde a la salida esperada del modelo sobre el conjunto de entrenamiento. Este valor representa la predicción que realizaría el modelo sobre un dato del cual no conoce ninguna de sus características.

Bajo este enfoque, la explicación de una predicción se expresa como una combinación lineal de contribuciones asociadas a cada variable, denominadas valores *SHAP*. Estas contribuciones se fundamentan en la teoría de juegos cooperativos, específicamente en los valores de *Shapley*. Dicha teoría distribuye de forma equitativa la ganancia total de un juego entre los distintos participantes en función de su contribución marginal al resultado final. En el contexto del aprendizaje automático, el "juego" corresponde a la predicción de una instancia concreta, mientras que los "jugadores" son las variables predictoras o características del modelo.

Las contribuciones o valores *SHAP* se interpretan de acuerdo a su signo y magnitud, de forma análoga a la interpretación de los coeficientes de la regresión logística. Sin embargo, mientras que en la regresión logística los coeficientes constituyen parámetros globales del modelo que describen la influencia general de cada característica sobre la salida, los valores *SHAP* proporcionan explicaciones específicas para cada instancia.

El signo del valor *SHAP* indica la dirección de la contribución que realiza el valor concreto de una variable en una observación determinada sobre la salida del modelo para una clase específica:

* $\phi > 0$. El valor que toma la variable en dicha observación contribuye a incrementar la salida asociada a la clase respecto al valor base del modelo, favoreciendo la predicción de esa categoría.

* $\phi < 0$. El valor que toma la variable contribuye a reducir la salida asociada a la clase respecto al valor base, disminuyendo la probabilidad asociada a dicha categoría.

La magnitud del valor *SHAP* representa la intensidad de esta contribución, indicando cuánto se desvía la salida del modelo respecto al valor base debido a la presencia de ese valor concreto de la variable.

Los valores *SHAP* permiten realizar tanto análisis de interpretabilidad global como local.

Desde una perspectiva global, la agregación de las contribuciones obtenidas para todas las instancias del conjunto de datos permite analizar qué características tienen mayor influencia en el comportamiento general del modelo. Además, estudiando la distribución de las contribuciones en función del valor concreto que toma cada característica, es posible determinar el impacto en dirección y magnitud sobre las distintas clases.

Desde una perspectiva local, los valores *SHAP* permiten explicar predicciones individuales, identificando qué variables han contribuido en mayor medida a que una observación concreta sea clasificada dentro de una determinada categoría.